In [ ]:
'''
git clone https://github.com/Superzchen/iFeature
'''

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, max_error, mean_absolute_error


In [2]:
df_fasta = pd.read_csv("pHopt_data.fasta")


In [11]:
import subprocess
import pandas as pd
import os

fasta_file = "pHopt_data.fasta"
feature_types = ["AAC", "DPC", "CTDC", "CTDT", "CTDD"]
dataframes = []

for f_type in feature_types:
    out_file = f"temp_{f_type}.tsv"
    print(f"⏳ Počítám fyzikálně-chemické vlastnosti pro: {f_type} ...")

    command = f"python iFeature/iFeature.py --file {fasta_file} --type {f_type} --out {out_file}"
    subprocess.run(command, shell=True, capture_output=True)
    if os.path.exists(out_file):
        df = pd.read_csv(out_file, sep='\t', index_col=0)
        dataframes.append(df)
        os.remove(out_file)
    else:
        print(f"❌ Pozor, nepodařilo se vygenerovat {f_type}. Zkontroluj terminál.")

super_baseline_df = pd.concat(dataframes, axis=1)

final_out = "pHopt_ifeatures.tsv"
super_baseline_df.to_csv(final_out, sep='\t')

print(f"✅ HOTOVO! Tvoje nová vylepšená tabulka je uložena jako: {final_out}")
print(f"📊 Konečný rozměr dat: {super_baseline_df.shape[0]} proteinů a ohromných {super_baseline_df.shape[1]} příznaků!")

⏳ Počítám fyzikálně-chemické vlastnosti pro: AAC ...
⏳ Počítám fyzikálně-chemické vlastnosti pro: DPC ...
⏳ Počítám fyzikálně-chemické vlastnosti pro: CTDC ...
⏳ Počítám fyzikálně-chemické vlastnosti pro: CTDT ...
⏳ Počítám fyzikálně-chemické vlastnosti pro: CTDD ...
✅ HOTOVO! Tvoje nová vylepšená tabulka je uložena jako: pHopt_ifeatures.tsv
📊 Konečný rozměr dat: 9855 proteinů a ohromných 693 příznaků!


In [12]:
X_baseline = pd.read_csv('pHopt_ifeatures.tsv', sep='\t', index_col=0)
df_original = pd.read_csv('pHopt_data.csv', index_col='Accession') 

In [13]:
train_mask = df_original['Split'].isin(['Training', 'Validation'])
test_mask = df_original['Split'] == 'Testing'

train_accessions = df_original[train_mask].index
test_accessions = df_original[test_mask].index

X_train_baseline = X_baseline.loc[X_baseline.index.isin(train_accessions)]
X_test_baseline = X_baseline.loc[X_baseline.index.isin(test_accessions)]

y_train_baseline = df_original.loc[X_train_baseline.index, 'pHopt']
y_test_baseline = df_original.loc[X_test_baseline.index, 'pHopt']

In [ ]:
rf_baseline = RandomForestRegressor(random_state=42)

param_grid_rf_baseline = {
    'max_depth': [10, 20, 30, None], 
    'max_features': ['sqrt', 'log2', 1.0], 
    'n_estimators': [100, 300, 500]
}

grid_search_rf_baseline = GridSearchCV(
    estimator=rf_baseline, 
    param_grid=param_grid_rf_baseline, 
    cv=5, 
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
)

grid_search_rf_baseline.fit(X_train_baseline, y_train_baseline)

best_rf_baseline = grid_search_rf_baseline.best_estimator_
y_pred_rf_baseline = best_rf_baseline.predict(X_test_baseline)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

In [7]:
mse_rf = mean_squared_error(y_test_baseline, y_pred_rf_baseline)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test_baseline, y_pred_rf_baseline)
max_err_rf = max_error(y_test_baseline, y_pred_rf_baseline)
r2_rf = r2_score(y_test_baseline, y_pred_rf_baseline)

r2_train_rf = r2_score(y_train_baseline, best_rf_baseline.predict(X_train_baseline))

print(f"Best params RF        : {grid_search_rf_baseline.best_params_}")
print(f"MSE (Test)            : {mse_rf:.3f}")
print(f"RMSE (Test)           : {rmse_rf:.3f}")
print(f"MAE (Test)            : {mae_rf:.3f}")
print(f"Max error (Test)      : {max_err_rf:.3f}")
print(f"R^2 (Test)            : {r2_rf:.3f}")
print(f"R^2 trénovacích dat   : {r2_train_rf:.3f}")

   VÝSLEDNÉ TESTOVACÍ METRIKY (RF iFeature)
Best params RF        : {'max_depth': 30, 'max_features': 'sqrt', 'n_estimators': 500}
MSE (Test)            : 0.910
RMSE (Test)           : 0.954
MAE (Test)            : 0.687
Max error (Test)      : 4.787
R^2 (Test)            : 0.317
R^2 trénovacích dat   : 0.902


In [ ]:
import subprocess
import os

fasta_file = "ligasa.fasta"
feature_types = ["AAC", "DPC", "CTDC", "CTDT", "CTDD"]
dataframes = []

for f_type in feature_types:
    out_file = f"temp_{f_type}.tsv"

    command = f"python iFeature/iFeature.py --file {fasta_file} --type {f_type} --out {out_file}"
    subprocess.run(command, shell=True, capture_output=True)
    if os.path.exists(out_file):
        df = pd.read_csv(out_file, sep='\t', index_col=0)
        dataframes.append(df)
        os.remove(out_file)
    else:
        print(f"❌") 


super_baseline_df = pd.concat(dataframes, axis=1)

final_out = "ligasa_ifeatures.tsv"
super_baseline_df.to_csv(final_out, sep='\t')

In [ ]:

nova_data_aac = pd.read_csv('ligasa_ifeatures.tsv', sep='\t', index_col=0)

odhadnute_pH = best_rf_baseline.predict(nova_data_aac)
print(f"pH = {odhadnute_pH[0]:.2f}")

In [ ]:
import subprocess
import os

fasta_file = "atpasa.fasta"
feature_types = ["AAC", "DPC", "CTDC", "CTDT", "CTDD"]
dataframes = []

for f_type in feature_types:
    out_file = f"temp_{f_type}.tsv"

    command = f"python iFeature/iFeature.py --file {fasta_file} --type {f_type} --out {out_file}"
    subprocess.run(command, shell=True, capture_output=True)
    if os.path.exists(out_file):
        df = pd.read_csv(out_file, sep='\t', index_col=0)
        dataframes.append(df)
        os.remove(out_file)
    else:
        print(f"❌") 


super_baseline_df = pd.concat(dataframes, axis=1)

final_out = "atpasa_ifeatures.tsv"
super_baseline_df.to_csv(final_out, sep='\t')

In [ ]:

nova_data_aac = pd.read_csv('atpasa_ifeatures.tsv', sep='\t', index_col=0)

odhadnute_pH = best_rf_baseline.predict(nova_data_aac)
print(f"pH={odhadnute_pH[0]:.2f}")